# Fase 2g: CT mixed-context ablation

Este notebook prueba una variante 2D mas conservadora que el mixed-context actual. La idea es mantener mas contexto anatomico: menos recortes, patches mas grandes y menor ponderacion positiva.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import config
from src.data.segmentation import build_ct_segmentation_dataframe, split_segmentation_dataframe
from src.training.segmentation_experiment import (
    get_device,
    make_segmentation_run_config,
    train_and_evaluate_segmentation,
    save_segmentation_artifacts,
    existing_segmentation_artifact,
    summarize_segmentation_results,
)

config.NUM_WORKERS = 0
device = get_device()
print(f'Device: {device}')

## 2. Configuracion principal

In [ ]:
run_config = make_segmentation_run_config(
    dataset_name='ct',
    architecture='attention_unet',
    run_mode='full',
    image_size=config.CT_IMAGE_SIZE,
    in_channels=1,
    variant_name='mixed30_patch192_pos70_tversky_pos10_thr080',
    loss_name='weighted_tversky_bce',
    bce_weight=0.2,
    dice_weight=0.8,
    pos_weight=10.0,
    tversky_alpha=0.3,
    tversky_beta=0.7,
    optimize_threshold=True,
    threshold_search_max=0.80,
    train_crop_size=(192, 192),
    train_crop_prob=0.3,
    positive_crop_prob=0.7,
    batch_size=8,
    epochs=24,
    early_stopping_patience=6,
    base_features=16,
)
run_config

## 3. Datos y splits

In [ ]:
ct_seg_df = build_ct_segmentation_dataframe(
    config.CT_DIR,
    config.CT_DIR / 'processed_segmentation_slices',
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)

train_df, val_df, test_df = split_segmentation_dataframe(
    ct_seg_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)

print({'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print({'train_studies': train_df.study_id.nunique(), 'val_studies': val_df.study_id.nunique(), 'test_studies': test_df.study_id.nunique()})

## 4. Entrenamiento

In [ ]:
seg_models_dir = config.MODELS_DIR / 'segmentation' / 'ct'
seg_results_dir = PROJECT_ROOT / 'results' / 'segmentation' / 'ct'

if existing_segmentation_artifact(run_config, seg_models_dir, seg_results_dir):
    print(f'Saltado: artefactos existentes detectados para {run_config.experiment_name}_{run_config.run_mode}.')
else:
    result = train_and_evaluate_segmentation(run_config, train_df, val_df, test_df, device)
    artifact_paths = save_segmentation_artifacts(run_config, result, seg_models_dir, seg_results_dir)
    print(artifact_paths)
    print(result['metrics'])
    print('threshold:', result['threshold'])

## 5. Comparacion CT

In [ ]:
summary_df = summarize_segmentation_results(sorted(seg_results_dir.glob('*_summary.json')))
cols = ['experiment', 'variant_name', 'loss_name', 'dice', 'iou', 'pixel_accuracy', 'threshold', 'hyperparameters']
summary_df[cols].sort_values('dice', ascending=False).reset_index(drop=True)

## 6. Hueco para segunda variante

Ejecutar esta variante solo si la configuracion principal se acerca o supera al mixed-context actual.

In [ ]:
candidate_config = make_segmentation_run_config(
    dataset_name='ct',
    architecture='attention_unet',
    run_mode='full',
    image_size=config.CT_IMAGE_SIZE,
    in_channels=1,
    variant_name='mixed20_patch192_pos60_tversky_pos10_thr080',
    loss_name='weighted_tversky_bce',
    bce_weight=0.2,
    dice_weight=0.8,
    pos_weight=10.0,
    tversky_alpha=0.3,
    tversky_beta=0.7,
    optimize_threshold=True,
    threshold_search_max=0.80,
    train_crop_size=(192, 192),
    train_crop_prob=0.2,
    positive_crop_prob=0.6,
    batch_size=8,
    epochs=24,
    early_stopping_patience=6,
    base_features=16,
)
candidate_config